In [ ]:
# ================================
# 07-test-set-evaluation.ipynb
# Evaluates all 108 previously trained PEFT adapters on the held-out test sets.
# ================================

# ------------------------------
# 1. Environment Setup
# ------------------------------
!pip uninstall -y torchao
!pip install -q peft --no-deps
!pip install -q accelerate

import torch, os, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------
# 2. Configuration
# ------------------------------
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 32

DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"
PRIMARY_RESULTS_ROOT = "/kaggle/input/notebooks/venkatkolluu/05-full-experiment-sweep"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the experiment list (108 runs)
try:
    sweep_df = pd.read_csv(f"{PRIMARY_RESULTS_ROOT}/experiment_results.csv")
    print(f"Loaded {len(sweep_df)} runs to evaluate.")
except Exception as e:
    print("Could not load experiment_results.csv. Make sure the dataset is attached in Kaggle.")
    sweep_df = pd.DataFrame()

# ------------------------------
# 3. Load Test Data
# ------------------------------
def build_test_loader(language):
    test_df = pd.read_parquet(f"{DATA_ROOT}/{language}/test.parquet")
    def tokenize(batch):
        return tokenizer(batch["premise"], batch["hypothesis"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    ds = Dataset.from_pandas(test_df).rename_column("label", "labels").map(tokenize, batched=True)
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return DataLoader(ds, batch_size=BATCH_SIZE)

loaders = {
    "hi": build_test_loader("hi"),
    "te": build_test_loader("te")
}

# ------------------------------
# 4. Evaluation Loop
# ------------------------------
results_file = "/kaggle/working/test_results.csv"
test_results = []

# Load base model ONCE to save massive time
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).cuda()
base_model.eval()

for idx, row in sweep_df.iterrows():
    method = row['method']
    lang = row['language']
    budget = row['budget']
    seed = row['seed']
    
    adapter_path = f"{PRIMARY_RESULTS_ROOT}/adapters/{method}/{lang}/budget{budget}_seed{seed}"
    print(f"[{idx+1}/{len(sweep_df)}] Evaluating {method} | {lang} | budget={budget} | seed={seed}")
    
    try:
        # Load adapter onto base model
        model = PeftModel.from_pretrained(base_model, adapter_path)
        
        preds, labels = [], []
        with torch.no_grad():
            for batch in loaders[lang]:
                batch = {k: v.cuda() for k, v in batch.items()}
                outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
                preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
                labels.extend(batch["labels"].cpu().numpy())
                
        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average="macro")
        
        # Unload adapter to restore base model state for next iteration
        model.unload()
        
        res = {
            "method": method, "language": lang, "budget": budget, "seed": seed,
            "test_accuracy": round(acc, 6), "test_macro_f1": round(f1, 6)
        }
        test_results.append(res)
        pd.DataFrame([res]).to_csv(results_file, mode='a', header=not os.path.exists(results_file), index=False)
        print(f"   -> Test Acc: {acc:.4f} | Test F1: {f1:.4f}")
        
    except Exception as e:
        print(f"   -> ERROR: {e}")

print("Test set evaluation complete.")

